# SPIN Baseline Comparison

Faithful reimplementation of SPIN (Sarkar et al., EMNLP 2025) using their exact `attentionSPIN.py` logic.

**Pipeline:**
1. Download Stage 4 image IDs + baseline captions from GitHub
2. Generate SPIN captions on the same 400 images
3. Score Baseline vs SPIN with CHAIR + bootstrap CIs

**SPIN hyperparameters (paper's CHAIR setting):**
- `routed_head = 0.8` — keep top 80% of heads per layer
- `small_num_mask = 0.1` — suppress bottom 20% by scaling to 0.1 (not zeroing)
- `start_layer = 0, end_layer = 32` — all layers

**Runtime:** A100 · ~30 min · no LoRA loaded

## 0. Install (run once, restart runtime, then skip)

In [1]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!pip install -q 'numpy==1.26.4'
!pip install -q 'transformers>=4.47' 'accelerate>=0.33' 'tokenizers>=0.21'
!pip install -q peft bitsandbytes safetensors 'torchao>=0.16.0'
!pip install -q pillow tqdm spacy sentencepiece
!python -m spacy download en_core_web_sm -q
print('Done. Runtime -> Restart session, then skip this cell.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 94.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is i

## 1. Imports + GPU check

In [4]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import json, gc, time, math, types, urllib.request
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    raise RuntimeError('No GPU — Runtime -> Change runtime type -> A100 GPU')
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/llava_hallucination_heads')
DRIVE.mkdir(parents=True, exist_ok=True)
(DRIVE / 'cache').mkdir(exist_ok=True)
(DRIVE / 'results').mkdir(exist_ok=True)

LOCAL = Path('/content/spin_work')
LOCAL.mkdir(exist_ok=True)
(LOCAL / 'images').mkdir(exist_ok=True)

GPU:  NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
Mounted at /content/drive


## 2. Download Stage 4 results from GitHub

In [2]:
S4_URL   = 'https://raw.githubusercontent.com/armaansandhu26/causal-grounding-lora/refs/heads/master/results/stage4_400img_results.json'
S4_LOCAL = LOCAL / 'stage4_400img_results.json'

if not S4_LOCAL.exists():
    print('Downloading Stage 4 results from GitHub...')
    urllib.request.urlretrieve(S4_URL, str(S4_LOCAL))

with open(S4_LOCAL) as f:
    s4 = json.load(f)

eval_images     = [r['img_id']  for r in s4['eval_captions']]
eval_gt_objects = [set(r['gt']) for r in s4['eval_captions']]
baseline_by_id  = {r['img_id']: r['captions']['baseline']
                   for r in s4['eval_captions']}

print(f'Images: {len(eval_images)}')
print(f'Baseline captions: {len(baseline_by_id)}')
print(f'\nStage 4 CHAIR results (from repo, for reference):')
for cond, v in s4['chair'].items():
    print(f'  {cond:10s}  CHAIRs={v["CHAIRs"]:.4f}  CHAIRi={v["CHAIRi"]:.4f}')

Images: 400
Baseline captions: 400

Stage 4 CHAIR results (from repo, for reference):
  baseline    CHAIRs=0.3625  CHAIRi=0.1375
  stage2      CHAIRs=0.0725  CHAIRi=0.0389
  stage3      CHAIRs=0.3500  CHAIRi=0.1344
  stage4      CHAIRs=0.0700  CHAIRi=0.0391


## 3. Download the 400 eval images

In [3]:
IMG_DIR = LOCAL / 'images'
img_id_to_path = {}
to_dl = []

for img_id in eval_images:
    fname = f'COCO_val2014_{img_id:012d}.jpg'
    p = IMG_DIR / fname
    img_id_to_path[img_id] = str(p)
    if not p.exists():
        to_dl.append((img_id, fname, p))

if to_dl:
    print(f'Downloading {len(to_dl)} images...')
    for img_id, fname, p in tqdm(to_dl, desc='Images'):
        try:
            urllib.request.urlretrieve(
                f'http://images.cocodataset.org/val2014/{fname}', str(p))
        except Exception as e:
            print(f'  Failed {img_id}: {e}')
else:
    print('All images already downloaded.')

print(f'Ready: {len(eval_images)} images')

Images:   0%|          | 0/400 [00:00<?, ?it/s]

Ready: 400 images


## 4. Load LLaVA-1.5-7B (base model only, no LoRA)

In [5]:
from transformers import AutoProcessor, LlavaForConditionalGeneration

MODEL_ID  = 'llava-hf/llava-1.5-7b-hf'
processor = AutoProcessor.from_pretrained(MODEL_ID)

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation='eager',
    device_map={'': 0},
)
model.eval()

text_cfg       = model.config.text_config
NUM_LAYERS     = text_cfg.num_hidden_layers
NUM_HEADS      = text_cfg.num_attention_heads
HEAD_DIM       = text_cfg.hidden_size // NUM_HEADS
IMAGE_TOKEN_ID = model.config.image_token_index
vision_cfg     = model.config.vision_config
NUM_IMG_TOKENS = (vision_cfg.image_size // vision_cfg.patch_size) ** 2
PROMPT         = 'USER: <image>\nDescribe this image in detail.\nASSISTANT:'

print(f'Model loaded. Layers={NUM_LAYERS}, Heads={NUM_HEADS}, HeadDim={HEAD_DIM}')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Model loaded. Layers=32, Heads=32, HeadDim=128
VRAM: 14.13 GB


## 5. SPIN generation

Exact replica of `attentionSPIN.py` from the SPIN repo (Sarkar et al., EMNLP 2025).
Uses method patching (same as original) instead of hooks.

In [7]:
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb


def get_visual_token_span(input_ids):
    ids  = input_ids[0]
    mask = (ids == IMAGE_TOKEN_ID)
    pos  = mask.nonzero(as_tuple=True)[0]
    n_ph = int(mask.sum().item())
    if n_ph >= NUM_IMG_TOKENS:
        return int(pos[0].item()), int(pos[-1].item()) + 1
    return int(pos[0].item()), int(pos[0].item()) + NUM_IMG_TOKENS


def llama_spin_forward(
    self, hidden_states, attention_mask=None, position_ids=None,
    past_key_value=None, output_attentions=False, use_cache=False, **kwargs):

    bsz, q_len, _ = hidden_states.size()

    # Newer transformers stores these on config, not directly on self
    num_heads   = self.config.num_attention_heads
    head_dim    = self.config.hidden_size // num_heads
    hidden_size = self.config.hidden_size

    query_states = self.q_proj(hidden_states).view(bsz, q_len, num_heads, head_dim).transpose(1, 2)
    key_states   = self.k_proj(hidden_states).view(bsz, q_len, num_heads, head_dim).transpose(1, 2)
    value_states = self.v_proj(hidden_states).view(bsz, q_len, num_heads, head_dim).transpose(1, 2)

    kv_seq_len = key_states.shape[-2]
    if past_key_value is not None:
        kv_seq_len += past_key_value.get_usable_length(kv_seq_len, self.layer_idx)

    cos, sin = self.rotary_emb(value_states, seq_len=kv_seq_len)
    query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin, position_ids)

    if past_key_value is not None:
        cache_kwargs = {'sin': sin, 'cos': cos}
        key_states, value_states = past_key_value.update(key_states, value_states, self.layer_idx, cache_kwargs)

    attn_weights = torch.matmul(query_states, key_states.transpose(2, 3)) / math.sqrt(head_dim)
    if attention_mask is not None:
        attn_weights = attn_weights + attention_mask
        attn_weights = torch.max(attn_weights, torch.tensor(torch.finfo(attn_weights.dtype).min))

    # ── SPIN gating ──
    num_routed_head = int(self.routed_head * num_heads)
    attn_scores = attn_weights.permute(0, 2, 1, 3)
    attn_scores_headwise = attn_scores[:, -1, :,
        self.img_start_idx:self.img_end_idx].sum(dim=-1).view(-1, num_heads)

    attn_score_std  = attn_scores_headwise.std(dim=1, keepdim=True)
    attn_score_norm = attn_scores_headwise / (attn_score_std + 1e-8)
    gates = F.softmax(attn_score_norm, dim=1)

    _, indices = torch.topk(gates, k=num_routed_head, dim=1)
    mask = F.one_hot(indices, num_classes=num_heads).sum(dim=1).to(query_states.dtype)
    mask[mask == 0] = self.small_num_mask

    if q_len > 1:
        mask = torch.cat([
            torch.ones((bsz * (q_len - 1), num_heads),
                       dtype=query_states.dtype, device=query_states.device),
            mask
        ], dim=0)
    mask = mask.reshape(bsz, q_len, -1)
    # ─────────────────

    attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)
    attn_output  = torch.matmul(attn_weights, value_states)
    attn_output  = attn_output.transpose(1, 2).reshape(bsz, q_len, num_heads, head_dim)
    attn_output  = torch.einsum('bne,bned->bned', mask, attn_output)
    attn_output  = attn_output.reshape(bsz, q_len, hidden_size)
    attn_output  = self.o_proj(attn_output)

    if not output_attentions:
        attn_weights = None
    return attn_output, attn_weights, past_key_value


@torch.no_grad()
def gen_spin(model_obj, image_path,
             start_layer=0, end_layer=32,
             routed_head=0.8, small_num_mask=0.1,
             max_new_tokens=80):
    """
    SPIN decoding.
    routed_head=0.8   → keep top 80% of heads (paper CHAIR setting)
    small_num_mask=0.1 → scale suppressed heads to 0.1 (not zero)
    """
    img    = Image.open(image_path).convert('RGB')
    inputs = processor(text=PROMPT, images=img,
                       return_tensors='pt').to(device, torch.float16)
    inputs['input_ids']      = inputs['input_ids'].long()
    inputs['attention_mask'] = inputs['attention_mask'].long()

    img_start, img_end = get_visual_token_span(inputs['input_ids'])

    # Patch self_attn forward on each decoder layer
    layers = model_obj.model.language_model.layers
    for i in range(start_layer, min(end_layer, len(layers))):
        attn = layers[i].self_attn
        attn.img_start_idx  = img_start
        attn.img_end_idx    = img_end
        attn.routed_head    = routed_head
        attn.small_num_mask = small_num_mask
        attn.forward        = types.MethodType(llama_spin_forward, attn)

    eos_id  = processor.tokenizer.eos_token_id
    past_kv = None
    cur_ids = inputs['input_ids']
    cur_msk = inputs['attention_mask']
    generated = []

    try:
        for _ in range(max_new_tokens):
            kw = dict(input_ids=cur_ids, attention_mask=cur_msk,
                      use_cache=True, past_key_values=past_kv,
                      return_dict=True)
            if past_kv is None:
                kw['pixel_values'] = inputs['pixel_values']
            out = model_obj(**kw)

            next_id = int(out.logits[:, -1, :].float().argmax(dim=-1).item())
            generated.append(next_id)
            if next_id == eos_id:
                break

            past_kv = out.past_key_values
            cur_ids = torch.tensor([[next_id]], dtype=torch.long, device=device)
            cur_msk = torch.cat([cur_msk,
                                 torch.ones(1, 1, dtype=torch.long, device=device)], dim=1)
    finally:
        # Restore original forward using the class's own method
        for i in range(start_layer, min(end_layer, len(layers))):
            attn = layers[i].self_attn
            for attr in ['img_start_idx', 'img_end_idx', 'routed_head', 'small_num_mask']:
                if hasattr(attn, attr):
                    delattr(attn, attr)
            attn.forward = types.MethodType(attn.__class__.forward, attn)

    return processor.tokenizer.decode(generated, skip_special_tokens=True)


# Sanity test
test_path = img_id_to_path[eval_images[0]]
print('Sanity test...')
cap = gen_spin(model, test_path)
torch.cuda.empty_cache()
print(f'  SPIN:     {cap[:120]}')
print(f'  Baseline: {baseline_by_id[eval_images[0]][:120]}')
print(f'VRAM after test: {torch.cuda.memory_allocated()/1e9:.2f} GB')
print('OK.')

Sanity test...


AttributeError: 'LlamaAttention' object has no attribute 'rotary_emb'

In [10]:
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb
import types, math
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image


def get_visual_token_span(input_ids):
    ids = input_ids[0]
    mask = (ids == IMAGE_TOKEN_ID)
    pos = mask.nonzero(as_tuple=True)[0]
    n_ph = int(mask.sum().item())

    if n_ph == 0:
        raise ValueError("No IMAGE_TOKEN_ID found in input_ids.")

    if n_ph >= NUM_IMG_TOKENS:
        return int(pos[0].item()), int(pos[-1].item()) + 1

    return int(pos[0].item()), int(pos[0].item()) + NUM_IMG_TOKENS


def repeat_kv(hidden_states, n_rep):
    """
    hidden_states: [bsz, num_kv_heads, seq_len, head_dim]
    returns:       [bsz, num_heads, seq_len, head_dim]
    """
    if n_rep == 1:
        return hidden_states

    bsz, num_kv_heads, slen, head_dim = hidden_states.shape
    hidden_states = hidden_states[:, :, None, :, :].expand(
        bsz, num_kv_heads, n_rep, slen, head_dim
    )
    return hidden_states.reshape(bsz, num_kv_heads * n_rep, slen, head_dim)


def llama_spin_forward(
    self,
    hidden_states,
    attention_mask=None,
    position_ids=None,
    past_key_value=None,
    output_attentions=False,
    use_cache=False,
    cache_position=None,
    position_embeddings=None,
    **kwargs,
):
    # Some newer transformers versions pass this as past_key_values
    if past_key_value is None and "past_key_values" in kwargs:
        past_key_value = kwargs["past_key_values"]

    bsz, q_len, _ = hidden_states.size()

    num_heads = self.config.num_attention_heads
    num_kv_heads = getattr(self.config, "num_key_value_heads", num_heads)
    num_kv_groups = num_heads // num_kv_heads

    head_dim = self.config.hidden_size // num_heads
    hidden_size = self.config.hidden_size

    query_states = self.q_proj(hidden_states)
    key_states = self.k_proj(hidden_states)
    value_states = self.v_proj(hidden_states)

    query_states = query_states.view(
        bsz, q_len, num_heads, head_dim
    ).transpose(1, 2)

    key_states = key_states.view(
        bsz, q_len, num_kv_heads, head_dim
    ).transpose(1, 2)

    value_states = value_states.view(
        bsz, q_len, num_kv_heads, head_dim
    ).transpose(1, 2)

    # New transformers path: decoder layer passes cos/sin directly
    if position_embeddings is not None:
        cos, sin = position_embeddings

    # Old transformers fallback
    elif hasattr(self, "rotary_emb"):
        kv_seq_len = key_states.shape[-2]

        if past_key_value is not None:
            try:
                kv_seq_len += past_key_value.get_usable_length(kv_seq_len, self.layer_idx)
            except Exception:
                pass

        cos, sin = self.rotary_emb(value_states, seq_len=kv_seq_len)

    else:
        raise RuntimeError(
            "No self.rotary_emb and no position_embeddings received. "
            "This is a transformers/LlamaAttention API mismatch."
        )

    # Your transformers version expects:
    # apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1)
    # So do NOT pass position_ids here.
    query_states, key_states = apply_rotary_pos_emb(
        query_states,
        key_states,
        cos,
        sin,
    )

    if past_key_value is not None:
        cache_kwargs = {
            "sin": sin,
            "cos": cos,
            "cache_position": cache_position,
        }

        try:
            key_states, value_states = past_key_value.update(
                key_states,
                value_states,
                self.layer_idx,
                cache_kwargs,
            )
        except TypeError:
            key_states, value_states = past_key_value.update(
                key_states,
                value_states,
                self.layer_idx,
            )

    key_states = repeat_kv(key_states, num_kv_groups)
    value_states = repeat_kv(value_states, num_kv_groups)

    attn_weights = torch.matmul(
        query_states,
        key_states.transpose(2, 3),
    ) / math.sqrt(head_dim)

    if attention_mask is not None:
        causal_mask = attention_mask[:, :, :, : key_states.shape[-2]]
        attn_weights = attn_weights + causal_mask

    # ── SPIN gating ──
    num_routed_head = max(1, int(self.routed_head * num_heads))

    img_start = int(self.img_start_idx)
    img_end = min(int(self.img_end_idx), attn_weights.shape[-1])

    attn_scores = attn_weights.permute(0, 2, 1, 3)

    attn_scores_headwise = attn_scores[:, -1, :, img_start:img_end].sum(dim=-1)
    attn_scores_headwise = attn_scores_headwise.view(-1, num_heads)

    attn_score_std = attn_scores_headwise.std(dim=1, keepdim=True)
    attn_score_norm = attn_scores_headwise / (attn_score_std + 1e-8)

    gates = F.softmax(attn_score_norm, dim=1)

    _, indices = torch.topk(gates, k=num_routed_head, dim=1)

    mask = F.one_hot(indices, num_classes=num_heads).sum(dim=1).to(query_states.dtype)
    mask[mask == 0] = self.small_num_mask

    if q_len > 1:
        prefix_mask = torch.ones(
            (bsz * (q_len - 1), num_heads),
            dtype=query_states.dtype,
            device=query_states.device,
        )
        mask = torch.cat([prefix_mask, mask], dim=0)

    mask = mask.reshape(bsz, q_len, num_heads)
    # ─────────────────

    attn_weights = nn.functional.softmax(
        attn_weights,
        dim=-1,
        dtype=torch.float32,
    ).to(query_states.dtype)

    attn_output = torch.matmul(attn_weights, value_states)

    attn_output = attn_output.transpose(1, 2).reshape(
        bsz,
        q_len,
        num_heads,
        head_dim,
    )

    attn_output = torch.einsum(
        "bnh,bnhd->bnhd",
        mask,
        attn_output,
    )

    attn_output = attn_output.reshape(bsz, q_len, hidden_size)
    attn_output = self.o_proj(attn_output)

    if not output_attentions:
        attn_weights = None

    # IMPORTANT:
    # Your LlamaDecoderLayer does:
    # hidden_states, _ = self.self_attn(...)
    # So we must return exactly 2 values.
    return attn_output, attn_weights


@torch.no_grad()
def gen_spin(
    model_obj,
    image_path,
    start_layer=0,
    end_layer=32,
    routed_head=0.8,
    small_num_mask=0.1,
    max_new_tokens=80,
):
    """
    SPIN decoding.
    routed_head=0.8      -> keep top 80% of heads
    small_num_mask=0.1   -> scale suppressed heads to 0.1
    """

    img = Image.open(image_path).convert("RGB")

    inputs = processor(
        text=PROMPT,
        images=img,
        return_tensors="pt",
    ).to(device, torch.float16)

    inputs["input_ids"] = inputs["input_ids"].long()
    inputs["attention_mask"] = inputs["attention_mask"].long()

    img_start, img_end = get_visual_token_span(inputs["input_ids"])

    layers = model_obj.model.language_model.layers
    patched_layers = range(start_layer, min(end_layer, len(layers)))

    # Patch self_attn forward on each decoder layer
    for i in patched_layers:
        attn = layers[i].self_attn

        if not hasattr(attn, "_spin_original_forward"):
            attn._spin_original_forward = attn.forward

        attn.img_start_idx = img_start
        attn.img_end_idx = img_end
        attn.routed_head = routed_head
        attn.small_num_mask = small_num_mask

        attn.forward = types.MethodType(llama_spin_forward, attn)

    eos_id = processor.tokenizer.eos_token_id

    past_kv = None
    cur_ids = inputs["input_ids"]
    cur_msk = inputs["attention_mask"]

    generated = []

    try:
        for _ in range(max_new_tokens):
            kw = dict(
                input_ids=cur_ids,
                attention_mask=cur_msk,
                use_cache=True,
                past_key_values=past_kv,
                return_dict=True,
            )

            if past_kv is None:
                kw["pixel_values"] = inputs["pixel_values"]

            out = model_obj(**kw)

            next_id = int(out.logits[:, -1, :].float().argmax(dim=-1).item())
            generated.append(next_id)

            if next_id == eos_id:
                break

            past_kv = out.past_key_values

            cur_ids = torch.tensor(
                [[next_id]],
                dtype=torch.long,
                device=device,
            )

            cur_msk = torch.cat(
                [
                    cur_msk,
                    torch.ones(
                        1,
                        1,
                        dtype=torch.long,
                        device=device,
                    ),
                ],
                dim=1,
            )

    finally:
        # Restore original forward
        for i in patched_layers:
            attn = layers[i].self_attn

            for attr in [
                "img_start_idx",
                "img_end_idx",
                "routed_head",
                "small_num_mask",
            ]:
                if hasattr(attn, attr):
                    delattr(attn, attr)

            if hasattr(attn, "_spin_original_forward"):
                attn.forward = attn._spin_original_forward
                delattr(attn, "_spin_original_forward")

    return processor.tokenizer.decode(generated, skip_special_tokens=True)


# Sanity test
test_path = img_id_to_path[eval_images[0]]

print("Sanity test...")

cap = gen_spin(model, test_path)

torch.cuda.empty_cache()

print(f"  SPIN:     {cap[:120]}")
print(f"  Baseline: {baseline_by_id[eval_images[0]][:120]}")
print(f"VRAM after test: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("OK.")

Sanity test...
  SPIN:     The image features a yellow fire hydrant situated on a sidewalk near a building. The fire hydrant is prominently display
  Baseline: The image features a yellow fire hydrant situated on a sidewalk next to a building. The fire hydrant is prominently plac
VRAM after test: 14.15 GB
OK.


## 6. Generate SPIN captions for all 400 images

Saves to Drive every 10 images. Resume-safe.

In [16]:
SPIN_CKPT = DRIVE / "cache" / "spin_captions_s4ids.json"

if SPIN_CKPT.exists():
    SPIN_CKPT.unlink()
    print(f"Deleted old checkpoint: {SPIN_CKPT}")
else:
    print(f"No checkpoint found at: {SPIN_CKPT}")

No checkpoint found at: /content/drive/MyDrive/llava_hallucination_heads/cache/spin_captions_s4ids.json


In [17]:
SPIN_CKPT = DRIVE / 'cache' / 'spin_captions_s4ids.json'

if SPIN_CKPT.exists():
    with open(SPIN_CKPT) as f:
        spin_caps = json.load(f)
    done_ids = {r['img_id'] for r in spin_caps}
    print(f'Resumed: {len(done_ids)}/{len(eval_images)} done')
else:
    spin_caps = []
    done_ids  = set()

t0         = time.time()
fail_count = 0

for img_id, gt_set in tqdm(zip(eval_images, eval_gt_objects),
                            total=len(eval_images), desc='SPIN'):
    if img_id in done_ids:
        continue

    caption = ''
    try:
        caption = gen_spin(model, img_id_to_path[img_id])
    except Exception as e:
        fail_count += 1
        print(f'  {img_id}: {e}')
        torch.cuda.empty_cache()
        gc.collect()

    spin_caps.append({'img_id': img_id, 'gt': list(gt_set), 'caption': caption})
    done_ids.add(img_id)

    if len(spin_caps) % 10 == 0:
        with open(SPIN_CKPT, 'w') as f:
            json.dump(spin_caps, f)
        torch.cuda.empty_cache()

with open(SPIN_CKPT, 'w') as f:
    json.dump(spin_caps, f)

print(f'\nDone: {len(spin_caps)} images in {(time.time()-t0)/60:.1f} min')
print(f'Failures: {fail_count}')
print(f'Saved: {SPIN_CKPT}')

SPIN:   0%|          | 0/400 [00:00<?, ?it/s]


Done: 400 images in 26.9 min
Failures: 0
Saved: /content/drive/MyDrive/llava_hallucination_heads/cache/spin_captions_s4ids.json


---
## Phase 2: Evaluation (no GPU needed)
---

## 7. COCO vocab + CHAIR scorer

In [5]:
import spacy
nlp = spacy.load('en_core_web_sm')

COCO_SYNONYMS = {
    'person':       ['man','woman','people','boy','girl','child','guy','lady',
                     'kid','baby','player','rider','skier','surfer','snowboarder'],
    'car':          ['vehicle','automobile','sedan','suv'],
    'dog':          ['puppy','dogs'],
    'cat':          ['kitten','cats'],
    'tv':           ['television','monitor','screen'],
    'couch':        ['sofa'],
    'cell phone':   ['phone','cellphone','smartphone'],
    'dining table': ['table','desk'],
    'wine glass':   ['glass'],
    'bicycle':      ['bike'],
    'motorcycle':   ['motorbike'],
    'airplane':     ['plane','jet'],
    'potted plant': ['plant'],
    'laptop':       ['computer'],
    'refrigerator': ['fridge'],
    'truck':        ['lorry'],
    'boat':         ['ship','sailboat'],
    'fire hydrant': ['hydrant'],
    'hot dog':      ['hotdog'],
    'traffic light':['stoplight'],
    'sports ball':  ['ball','football','soccer ball','basketball'],
    'baseball bat': ['bat'],
    'tennis racket':['racket','racquet'],
}
MULTIWORD = {
    'hydrant':  'fire hydrant',
    'hotdog':   'hot dog',
    'stoplight':'traffic light',
    'bat':      'baseball bat',
    'racket':   'tennis racket',
    'racquet':  'tennis racket',
}
ALL_COCO = [
    'person','bicycle','car','motorcycle','airplane','bus','train','truck','boat',
    'traffic light','fire hydrant','stop sign','parking meter','bench',
    'bird','cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe',
    'backpack','umbrella','handbag','tie','suitcase','frisbee','skis','snowboard',
    'sports ball','kite','baseball bat','baseball glove','skateboard','surfboard',
    'tennis racket','bottle','wine glass','cup','fork','knife','spoon','bowl',
    'banana','apple','sandwich','orange','broccoli','carrot','hot dog','pizza',
    'donut','cake','chair','couch','potted plant','bed','dining table','toilet',
    'tv','laptop','mouse','remote','keyboard','cell phone','microwave','oven',
    'toaster','sink','refrigerator','book','clock','vase','scissors',
    'teddy bear','hair drier','toothbrush',
]
OBJECT_VOCAB = set(ALL_COCO)
for syns in COCO_SYNONYMS.values(): OBJECT_VOCAB.update(syns)
OBJECT_VOCAB.update(MULTIWORD.keys())


def find_content_words(caption, gt_objects):
    gt_norm     = {o.lower() for o in gt_objects}
    expanded_gt = set(gt_norm)
    for canonical, syns in COCO_SYNONYMS.items():
        if canonical in gt_norm: expanded_gt.update(syns)
    for alias, canonical in MULTIWORD.items():
        if canonical in gt_norm: expanded_gt.add(alias)
    doc = nlp(caption)
    obj_words, hall_words = [], []
    for tok in doc:
        w = tok.text.lower().strip()
        if tok.pos_ not in ('NOUN', 'PROPN') or len(w) < 2: continue
        canonical = MULTIWORD.get(w, w)
        if w in OBJECT_VOCAB or canonical in OBJECT_VOCAB:
            obj_words.append(w)
            if w not in expanded_gt and canonical not in expanded_gt:
                hall_words.append(w)
    return obj_words, hall_words


def score_chair(captions_gt):
    chairs_list, chairi_list = [], []
    for cap, gt in captions_gt:
        if not cap: continue
        obj_w, hall_w = find_content_words(cap, gt)
        chairs_list.append(1 if hall_w else 0)
        chairi_list.append(len(hall_w) / max(len(obj_w), 1))
    return (float(np.mean(chairs_list)) if chairs_list else 0.0,
            float(np.mean(chairi_list)) if chairi_list else 0.0)


print('CHAIR scorer ready.')

CHAIR scorer ready.


## 8. Results: Baseline vs SPIN + Bootstrap CIs

In [8]:
# Load SPIN captions
with open(DRIVE / 'cache' / 'spin_captions_s4ids.json') as f:
    spin_caps = json.load(f)

# Load Stage 4 baseline captions directly from GitHub
S4_URL = 'https://raw.githubusercontent.com/armaansandhu26/causal-grounding-lora/refs/heads/master/results/stage4_400img_results.json'
import urllib.request, json as _json
s4_raw = urllib.request.urlopen(S4_URL).read()
s4 = _json.loads(s4_raw)

baseline_by_id = {r['img_id']: {'caption': r['captions']['baseline'],
                                  'gt':      set(r['gt'])}
                  for r in s4['eval_captions']}

# Build paired records
paired = []
for r in spin_caps:
    iid = r['img_id']
    if iid not in baseline_by_id:
        continue
    paired.append({
        'img_id':       iid,
        'gt':           baseline_by_id[iid]['gt'],
        'baseline_cap': baseline_by_id[iid]['caption'],
        'spin_cap':     r['caption'],
    })

# print(f'Paired images: {len(paired)} / {len(eval_images)}')
# if len(paired) < len(eval_images):
#     print(f'  Warning: {len(eval_images) - len(paired)} images not matched')

# Sample outputs
print('\n--- Sample outputs ---')
for p in paired[:3]:
    print(f'  [{p["img_id"]}] GT: {sorted(p["gt"])[:4]}')
    print(f'    Baseline: {p["baseline_cap"][:100]}')
    print(f'    SPIN:     {p["spin_cap"][:100]}')
    print()


# Bootstrap CI
def bootstrap_ci(values, n_boot=2000, seed=42):
    rng  = np.random.RandomState(seed)
    arr  = np.array(values)
    boot = [arr[rng.randint(0, len(arr), len(arr))].mean() for _ in range(n_boot)]
    return float(arr.mean()), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))


# Score both conditions
results = {}
for method, cap_key in [('Baseline', 'baseline_cap'), ('SPIN (Sarkar+25)', 'spin_cap')]:
    chair_pairs = [(p[cap_key], p['gt']) for p in paired if p[cap_key]]
    chairs, chairi = score_chair(chair_pairs)
    lengths = [len(p[cap_key].split()) for p in paired if p[cap_key]]

    per_img_s, per_img_i = [], []
    for p in paired:
        cap = p[cap_key]
        if not cap: continue
        obj_w, hall_w = find_content_words(cap, p['gt'])
        per_img_s.append(1 if hall_w else 0)
        per_img_i.append(len(hall_w) / max(len(obj_w), 1))

    cs_m, cs_lo, cs_hi = bootstrap_ci(per_img_s)
    ci_m, ci_lo, ci_hi = bootstrap_ci(per_img_i)

    results[method] = {
        'CHAIRs':    chairs,  'CHAIRs_CI': (cs_lo, cs_hi),
        'CHAIRi':    chairi,  'CHAIRi_CI': (ci_lo, ci_hi),
        'avg_len':   float(np.mean(lengths)),
        'n':         len(chair_pairs),
    }

# Print table
print('='*80)
print(f'{"Method":<22} {"CHAIRs":>8} {"95% CI":>18}  '
      f'{"CHAIRi":>8} {"95% CI":>18} {"AvgLen":>7}')
print('-'*80)

base_cs = results['Baseline']['CHAIRs']
for method, r in results.items():
    cs_lo, cs_hi = r['CHAIRs_CI']
    ci_lo, ci_hi = r['CHAIRi_CI']
    delta = '' if method == 'Baseline' else \
            f'  ({(base_cs - r["CHAIRs"]) / base_cs * 100:+.1f}%)'
    print(f'{method:<22} {r["CHAIRs"]:>8.3f} [{cs_lo:.3f}, {cs_hi:.3f}]  '
          f'{r["CHAIRi"]:>8.3f} [{ci_lo:.3f}, {ci_hi:.3f}] {r["avg_len"]:>7.1f}{delta}')

print('='*80)
print(f'\nn = {len(paired)} paired images')
print(f'SPIN config: routed_head=0.8, small_num_mask=0.1, layers 0-32')

print('\n--- Stage 4 repo numbers (for reference) ---')
for cond, v in s4['chair'].items():
    print(f'  {cond:10s}  CHAIRs={v["CHAIRs"]:.4f}  CHAIRi={v["CHAIRi"]:.4f}')


--- Sample outputs ---
  [293474] GT: ['book', 'fire hydrant']
    Baseline: The image features a yellow fire hydrant situated on a sidewalk next to a building. The fire hydrant
    SPIN:     The image features a yellow fire hydrant situated on a sidewalk near a building. The fire hydrant is

  [465878] GT: ['person', 'surfboard']
    Baseline: The image captures a man skillfully riding a surfboard on a wave in the ocean. He is positioned in t
    SPIN:     The image captures a man skillfully riding a wave on a surfboard in the ocean. The surfer is in the 

  [419401] GT: ['bicycle', 'train']
    Baseline: The image features a red train with a bicycle symbol on the side. The train is parked at a station, 
    SPIN:     The image features a red train with a bicycle symbol on the side of the train. The bicycle symbol is

Method                   CHAIRs             95% CI    CHAIRi             95% CI  AvgLen
--------------------------------------------------------------------------------

## 9. Save results

In [9]:
out = {
    'n_paired':   len(paired),
    'spin_config': {
        'start_layer':    0,
        'end_layer':      32,
        'routed_head':    0.8,
        'small_num_mask': 0.1,
        'note': 'Paper CHAIR setting. Faithful replica of attentionSPIN.py.'
    },
    'results': {
        k: {kk: list(vv) if isinstance(vv, tuple) else vv
            for kk, vv in v.items()}
        for k, v in results.items()
    }
}

out_path = DRIVE / 'results' / 'stage5_spin_comparison.json'
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2)
print(f'Saved: {out_path}')

Saved: /content/drive/MyDrive/llava_hallucination_heads/results/stage5_spin_comparison.json
